<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/text2vid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text-to-video : générer une courte vidéo à partir d'un texte

Dans ce notebook, on va générer une **courte vidéo** à partir d'un **prompt textuel**.

## Idée principale
On ne génère plus seulement une image cohérente avec un texte.
On génère une **suite d'images cohérentes dans le temps**.

## Ce qu'on veut comprendre
- ce qu'est un modèle text-to-video ;
- pourquoi c'est plus difficile que le text-to-image ;
- comment régler la longueur et la qualité d'une vidéo générée.

## Pourquoi est-ce plus difficile que le text-to-image ?

En text-to-image, une seule image doit être plausible.

En text-to-video, il faut :
- une image plausible à chaque frame ;
- et une cohérence temporelle entre les frames.

## Risques fréquents
- mouvement instable ;
- changements d'apparence ;
- scintillement ;
- objets qui se déforment.

In [ ]:
!nvidia-smi || true

import sys
import platform
import torch

print("Python :", sys.version)
print("Plateforme :", platform.platform())
print("Torch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## Installer les bibliothèques utiles

Nous allons utiliser :
- `diffusers`
- `transformers`
- `accelerate`
- `imageio[ffmpeg]`

## Important
Les modèles vidéo sont lourds.
On utilisera donc plusieurs optimisations mémoire.

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors imageio imageio-ffmpeg

## Charger la pipeline text-to-video

Nous utilisons ici une pipeline `TextToVideoSDPipeline`.

### Modèle choisi
`ali-vilab/text-to-video-ms-1.7b`

C'est un modèle text-to-video connu et documenté dans Diffusers. Honnêtement je ne sais pas si la GPU supportera.

In [ ]:
import torch
from diffusers import TextToVideoSDPipeline

model_id = "ali-vilab/text-to-video-ms-1.7b"

pipe = TextToVideoSDPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    variant="fp16" if torch.cuda.is_available() else None
)

## Activer les optimisations mémoire

Les docs Diffusers recommandent plusieurs stratégies pour réduire l'usage mémoire :
- CPU offloading
- VAE slicing
- feed-forward chunking

Nous les activons ici pour augmenter les chances que cela tourne sur Colab.

In [ ]:
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.unet.enable_forward_chunking(chunk_size=1, dim=1)

print("Optimisations mémoire activées.")

## Choisir un prompt

Pour une première démonstration, mieux vaut un prompt simple et visuel.

### Conseils
- sujet clair ;
- action simple ;
- décor simple ;
- style explicite.

In [ ]:
prompt = (
    "A small brown teddy bear riding a skateboard in a city park, "
    "full body visible, centered subject, subject remains in frame, "
    "static camera, medium wide shot, smooth motion, cinematic"
)

negative_prompt = (
    "blurry, distorted, low quality, flickering, extra limbs, "
    "cropped subject, out of frame, off-center subject, camera pan, camera zoom"
)

## Paramètres de génération

Pour limiter la mémoire et le temps de calcul, on choisit ici :
- peu de frames ;
- un nombre d'étapes modéré ;
- une guidance raisonnable.

### Important
Plus il y a de frames et de steps, plus c'est coûteux.

In [ ]:
seed = 42
num_frames = 16
num_inference_steps = 30
guidance_scale = 10

generator = torch.Generator(device="cpu").manual_seed(seed)

print("seed =", seed)
print("num_frames =", num_frames)
print("num_inference_steps =", num_inference_steps)
print("guidance_scale =", guidance_scale)

## Générer les frames

Le modèle renvoie une liste d'images correspondant aux différentes frames de la vidéo.

In [ ]:
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=num_frames,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator
)

frames = result.frames[0]
print("Nombre de frames générées :", len(frames))

## Visualiser quelques frames

Avant d'exporter la vidéo, on regarde quelques images générées.

In [ ]:
import matplotlib.pyplot as plt

indices = [0, len(frames)//3, 2*len(frames)//3, len(frames)-1]

plt.figure(figsize=(14, 4))
for i, idx in enumerate(indices, start=1):
    plt.subplot(1, 4, i)
    plt.imshow(frames[idx])
    plt.title(f"frame {idx}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Exporter les frames en vidéo MP4

Nous allons maintenant assembler les frames en une courte vidéo.

In [ ]:
from diffusers.utils import export_to_video

video_path = export_to_video(frames, output_video_path="/content/text_to_video_demo.mp4")
print("Vidéo exportée :", video_path)

In [ ]:
from IPython.display import Video, display
display(Video("/content/text_to_video_demo.mp4", embed=True, width=640))